In [1]:
import numpy as np
from scipy.spatial.distance import cosine
from scipy.stats import weibull_min
from scipy.special import softmax


def eucos_distance(x, y, eu_scale=1.0, eu_weight=1.0, cos_weight=1.0, eps=1e-12):
    eu = np.linalg.norm(x - y) / max(eu_scale, eps)
    nx = np.linalg.norm(x)
    ny = np.linalg.norm(y)
    if nx < eps or ny < eps:
        cos_dist = 1.0
    else:
        cos_dist = cosine(x, y)
    return eu_weight * eu + cos_weight * cos_dist


class OpenMaxLogits:
    def __init__(self, tail_size=20, alpha=10, eu_weight=1.0, cos_weight=1.0):
        self.tail_size = tail_size
        self.alpha = alpha
        self.eu_weight = eu_weight
        self.cos_weight = cos_weight
        self.mavs = None
        self.weibulls = None
        self.eu_scales = None
        self.classes_ = None

    def fit(self, train_logits, train_true, train_pred):
        train_logits = np.asarray(train_logits, dtype=np.float64)
        train_true = np.asarray(train_true)
        train_pred = np.asarray(train_pred)

        self.classes_ = np.unique(train_true)
        n_classes = len(self.classes_)

        if train_logits.shape[1] != n_classes:
            raise ValueError("train_logits second dimension must equal number of classes")

        self.mavs = np.zeros((n_classes, n_classes), dtype=np.float64)
        self.weibulls = {}
        self.eu_scales = np.ones(n_classes, dtype=np.float64)

        for c in self.classes_:
            mask = (train_true == c) & (train_pred == c)
            class_logits = train_logits[mask]

            if len(class_logits) < max(5, self.tail_size):
                raise ValueError(f"Not enough correctly classified examples for class {c}")

            mav = class_logits.mean(axis=0)
            self.mavs[c] = mav

            eu_dists = np.array([np.linalg.norm(z - mav) for z in class_logits], dtype=np.float64)
            eu_scale = np.median(eu_dists) + 1e-12
            self.eu_scales[c] = eu_scale

            dists = np.array([
                eucos_distance(
                    z, mav,
                    eu_scale=eu_scale,
                    eu_weight=self.eu_weight,
                    cos_weight=self.cos_weight
                )
                for z in class_logits
            ], dtype=np.float64)

            tail = np.sort(dists)[-self.tail_size:]
            shape, loc, scale = weibull_min.fit(tail, floc=0.0)
            self.weibulls[c] = (shape, loc, scale)

        return self

    def _weibull_cdf(self, c, dist):
        shape, loc, scale = self.weibulls[c]
        return weibull_min.cdf(dist, c=shape, loc=loc, scale=scale)

    def recalibrate(self, x):
        """
        x: shape (n_classes,)
        returns:
            openmax_probs: shape (n_classes + 1,)  # [unknown, class0, class1, ...]
            revised_logits: shape (n_classes,)
            unknown_logit: float
        """
        x = np.asarray(x, dtype=np.float64)
        n_classes = len(self.classes_)

        ranked = np.argsort(x)[::-1]
        topk = ranked[:min(self.alpha, n_classes)]

        omega = np.ones(n_classes, dtype=np.float64)

        for rank_pos, c in enumerate(topk):
            alpha_weight = (self.alpha - rank_pos) / self.alpha
            dist = eucos_distance(
                x, self.mavs[c],
                eu_scale=self.eu_scales[c],
                eu_weight=self.eu_weight,
                cos_weight=self.cos_weight
            )
            wscore = self._weibull_cdf(c, dist)
            omega[c] = 1.0 - alpha_weight * wscore

        revised_logits = x * omega
        unknown_logit = np.sum(x * (1.0 - omega))

        logits_with_unknown = np.concatenate([[unknown_logit], revised_logits])
        openmax_probs = softmax(logits_with_unknown)

        return openmax_probs, revised_logits, unknown_logit

    def predict_proba(self, test_logits):
        test_logits = np.asarray(test_logits, dtype=np.float64)
        probs = np.array([self.recalibrate(x)[0] for x in test_logits])
        return probs

    def predict(self, test_logits, threshold=None):
        probs = self.predict_proba(test_logits)
        preds = []

        for p in probs:
            idx = np.argmax(p)
            best_prob = p[idx]

            if idx == 0:
                preds.append(-1)  # unknown
            else:
                pred_class = idx - 1
                if threshold is not None and best_prob < threshold:
                    preds.append(-1)
                else:
                    preds.append(pred_class)

        return np.array(preds), probs

In [4]:
import os
import sys
import pandas as pd
module_path = "../osr_tiny/outputs_osr/"
if module_path not in sys.path:
    sys.path.append(module_path)

OUT_DIR = "../osr_tiny/outputs_osr"

In [3]:
train_logit_path_cifar =r'cifar10_train_logit_vit.csv'
test_id_logit_path_cifar = r'cifar10_test_logit_vit.csv'
test_ood_logit_path_cifar = r'cifar100_test_logit_vit.csv'

train_logit_path_mnist = r'mnist_train_id_0_5_logit_cnn.csv'
test_id_logit_path_mnist = r'mnist_test_id_0_5_logit_cnn.csv'
test_ood_logit_path_mnist = r'mnist_test_ood_6_9_logit_cnn.csv'

train_logit_path_svhn = r'svhn_train_id_logit_wrn.csv'
test_id_logit_path_svhn = r'svhn_test_id_logit_wrn.csv'
test_ood_logit_path_svhn = r'svhn_test_ood_logit_wrn.csv'

train_logit_path_mp10 = r'MP10_class_logits_training.csv'
test_logit_path_mp10 = r'MP10_class_logits_testing_w_open.csv'

train_logit_path_tiny = os.path.join(OUT_DIR, 'vgg32_train_logits.csv')
test_logit_path_tiny = os.path.join(OUT_DIR, 'vgg32_val_logits.csv')

train_logit_path_cifar_first6 = r'cifar10_train_id_first6_logit_vit.csv'
test_id_logit_path_cifar_first6 = r'cifar10_test_id_first6_logit_vit.csv'
test_ood_logit_path_cifar_first6 = r'cifar10_test_ood_last4_logit_vit.csv'

In [22]:
print(pd.read_csv(test_logit_path_mp10).columns)

Index(['Instance Name', ' Real Class', 'bppif', 'cwlp-ss', 'bpp', 'scheduling',
       'cvrp', 'clp', 'inr', 'lotsizing', 'coloring', 'tup'],
      dtype='object')


In [23]:
#example_df = pd.read_csv(train_logit_path_svhn)
#print(example_df.head())

In [24]:
#print(pd.read_csv(train_logit_path_cifar).head())
#print(pd.read_csv(train_logit_path_mnist).head())
#print(pd.read_csv(train_logit_path_svhn).head())
#print(pd.read_csv(train_logit_path_mp10).head())
#print(pd.read_csv(train_logit_path_tiny).head())

In [8]:
import os
import json
import pandas as pd
import numpy as np

# =========================================================
# paths
# =========================================================
train_logit_path_cifar = r'cifar10_train_logit_vit.csv'
test_id_logit_path_cifar = r'cifar10_test_logit_vit.csv'
test_ood_logit_path_cifar = r'cifar100_test_logit_vit.csv'

train_logit_path_mnist = r'mnist_train_id_0_5_logit_cnn.csv'
test_id_logit_path_mnist = r'mnist_test_id_0_5_logit_cnn.csv'
test_ood_logit_path_mnist = r'mnist_test_ood_6_9_logit_cnn.csv'

train_logit_path_svhn = r'svhn_train_id_logit_wrn.csv'
test_id_logit_path_svhn = r'svhn_test_id_logit_wrn.csv'
test_ood_logit_path_svhn = r'svhn_test_ood_logit_wrn.csv'

train_logit_path_mp10 = r'MP10_class_logits_training.csv'
test_logit_path_mp10 = r'MP10_class_logits_testing_w_open.csv'


train_logit_path_cifar_first6 = r'cifar10_train_id_first6_logit_vit.csv'
test_id_logit_path_cifar_first6 = r'cifar10_test_id_first6_logit_vit.csv'
test_ood_logit_path_cifar_first6 = r'cifar10_test_ood_last4_logit_vit.csv'

OUT_DIR = "../osr_tiny/outputs_osr"
train_logit_path_tiny = os.path.join(OUT_DIR, 'vgg32_train_logits.csv')
test_logit_path_tiny = os.path.join(OUT_DIR, 'vgg32_val_logits.csv')

SAVE_DIR = "processed_logits"
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================================================
# MP canonical class order
# =========================================================
MP_LABEL_ORDER = [
    "bpp", "bppif", "clp", "coloring", "cvrp", "cwl", "inr", "lotsizing",
    "scheduling", "tup", "bpp2", "cpmp", "cuttingstock", "gap", "kps",
    "maplabeling", "pcp", "relaxedClique", "vrptw"
]

# aliases from file names / column names to canonical names
MP_NAME_ALIASES = {
    "cwlp-ss": "cwl",
    "cwl": "cwl",
}

MP_CLASS_TO_IDX = {name: i for i, name in enumerate(MP_LABEL_ORDER)}

# =========================================================
# helpers
# =========================================================
def save_no_header(df: pd.DataFrame, path: str) -> None:
    """
    Save dataframe without header for consistent downstream loading.
    """
    df.to_csv(path, index=False, header=False)


def show_preview(name: str, df: pd.DataFrame, n: int = 5) -> None:
    print(f"\n{name}")
    print("shape:", df.shape)
    print(df.head(n))


def sanity_check_no_nan(df: pd.DataFrame, name: str) -> None:
    if df.isna().any().any():
        raise ValueError(f"{name} contains NaN values after preprocessing.")
    print(f"{name}: no NaN values found.")


def sanity_check_same_width(train_df: pd.DataFrame, test_df: pd.DataFrame, name: str) -> None:
    if train_df.shape[1] != test_df.shape[1]:
        raise ValueError(
            f"{name}: train/test processed files do not have same number of columns.\n"
            f"train: {train_df.shape[1]}, test: {test_df.shape[1]}"
        )
    print(f"{name}: train/test width check passed ({train_df.shape[1]} columns).")


# =========================================================
# generic datasets: headerless [label, logits...]
# CIFAR / MNIST / SVHN
# =========================================================
def read_headerless_logits(path: str) -> pd.DataFrame:
    """
    Read a csv with no header where the first column is label and the
    remaining columns are logits.
    """
    df = pd.read_csv(path, header=None)
    return df

'''
def combine_headerless_test_files(test_id_path: str, test_ood_path: str) -> pd.DataFrame:
    df_id = read_headerless_logits(test_id_path)
    df_ood = read_headerless_logits(test_ood_path)

    if df_id.shape[1] != df_ood.shape[1]:
        raise ValueError(
            f"Column mismatch when combining:\n"
            f"{test_id_path}: {df_id.shape[1]} cols\n"
            f"{test_ood_path}: {df_ood.shape[1]} cols"
        )

    combined = pd.concat([df_id, df_ood], axis=0, ignore_index=True)
    return combined
'''
def combine_headerless_test_files(
    test_id_path: str,
    test_ood_path: str,
    ood_label_offset: int | None = None,
) -> pd.DataFrame:
    df_id = read_headerless_logits(test_id_path)
    df_ood = read_headerless_logits(test_ood_path)

    if df_id.shape[1] != df_ood.shape[1]:
        raise ValueError(
            f"Column mismatch when combining:\n"
            f"{test_id_path}: {df_id.shape[1]} cols\n"
            f"{test_ood_path}: {df_ood.shape[1]} cols"
        )

    df_ood = df_ood.copy()

    if ood_label_offset is not None:
        df_ood.iloc[:, 0] = df_ood.iloc[:, 0].astype(int) + int(ood_label_offset)

    combined = pd.concat([df_id, df_ood], axis=0, ignore_index=True)
    return combined

# =========================================================
# Tiny dataset: headered [label, logit_0, logit_1, ...]
# =========================================================
def preprocess_tiny(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    if "label" not in df.columns:
        raise ValueError(f"'label' column not found in Tiny file: {path}")

    logit_cols = [c for c in df.columns if c.startswith("logit_")]
    if not logit_cols:
        raise ValueError(f"No logit_ columns found in Tiny file: {path}")

    logit_cols = sorted(logit_cols, key=lambda x: int(x.split("_")[1]))
    out = df[["label"] + logit_cols].copy()
    return out


# =========================================================
# MP dataset
# output format:
#   label, logit_0, logit_1, ...
#
# metadata JSON saves:
#   - canonical class order
#   - canonical label -> integer id
#   - raw logit columns in file order
#   - canonicalized logit columns in file order
#   - global label id for each logit_i
# =========================================================
def canonicalize_mp_name(name: str) -> str:
    name = str(name).strip()
    return MP_NAME_ALIASES.get(name, name)


def preprocess_mp(path: str):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    required_cols = ["Instance Name", "Real Class"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(
                f"Expected columns {required_cols} in {path}, "
                f"but got columns: {df.columns.tolist()}"
            )

    # canonicalize real class
    real_class = df["Real Class"].astype(str).str.strip().map(canonicalize_mp_name)

    unknown_real_classes = sorted(set(real_class) - set(MP_CLASS_TO_IDX))
    if unknown_real_classes:
        raise ValueError(
            f"These Real Class values are not in MP_LABEL_ORDER:\n{unknown_real_classes}"
        )

    labels = real_class.map(MP_CLASS_TO_IDX).astype(int)

    # everything except metadata is treated as a logit column
    raw_logit_cols = [c for c in df.columns if c not in ["Instance Name", "Real Class"]]
    canonical_logit_names = [canonicalize_mp_name(c) for c in raw_logit_cols]

    unknown_logit_cols = sorted(set(canonical_logit_names) - set(MP_CLASS_TO_IDX))
    if unknown_logit_cols:
        raise ValueError(
            f"These MP logit columns are not in MP_LABEL_ORDER:\n{unknown_logit_cols}"
        )

    logits = df[raw_logit_cols].copy()
    logits.columns = [f"logit_{i}" for i in range(len(raw_logit_cols))]

    out = pd.concat([labels.rename("label"), logits], axis=1)

    metadata = {
        "source_file": path,
        "num_rows": int(len(df)),
        "num_logits": int(len(raw_logit_cols)),
        "canonical_label_order": MP_LABEL_ORDER,
        "canonical_label_to_id": MP_CLASS_TO_IDX,
        "raw_logit_columns_in_file_order": raw_logit_cols,
        "canonical_logit_columns_in_file_order": canonical_logit_names,
        "logit_index_to_canonical_label_id": {
            f"logit_{i}": int(MP_CLASS_TO_IDX[name])
            for i, name in enumerate(canonical_logit_names)
        },
        "logit_index_to_canonical_label_name": {
            f"logit_{i}": name
            for i, name in enumerate(canonical_logit_names)
        },
    }

    return out, metadata


def save_json(obj: dict, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


# =========================================================
# main processing
# =========================================================
def main():
    # -------------------------
    # CIFAR
    # -------------------------
    cifar_train = read_headerless_logits(train_logit_path_cifar)

    # number of ID classes from train logits
    cifar_id_classes = cifar_train.shape[1] - 1

    cifar_test = combine_headerless_test_files(
        test_id_logit_path_cifar,
        test_ood_logit_path_cifar,
        ood_label_offset=cifar_id_classes
    )

    save_no_header(cifar_train, os.path.join(SAVE_DIR, "cifar_train_processed.csv"))
    save_no_header(cifar_test, os.path.join(SAVE_DIR, "cifar_test_combined_processed.csv"))

    show_preview("CIFAR train", cifar_train)
    show_preview("CIFAR test combined", cifar_test)

    #first 6
    cifar_train_first6 = read_headerless_logits(train_logit_path_cifar_first6)

    # number of ID classes from train logits
    #cifar_id_classes = cifar_train.shape[1] - 1

    cifar_test_first6 = combine_headerless_test_files(
        test_id_logit_path_cifar_first6 ,
        test_ood_logit_path_cifar_first6 
        #ood_label_offset=cifar_id_classes
    )

    save_no_header(cifar_train, os.path.join(SAVE_DIR, "cifar_train_first6_processed.csv"))
    save_no_header(cifar_test, os.path.join(SAVE_DIR, "cifar_test_first6_combined_processed.csv"))

    show_preview("CIFAR first6 train", cifar_train_first6 )
    show_preview("CIFAR first6 test combined", cifar_test_first6 )

    # -------------------------
    # MNIST
    # -------------------------
    mnist_train = read_headerless_logits(train_logit_path_mnist)
    mnist_test = combine_headerless_test_files(
        test_id_logit_path_mnist,
        test_ood_logit_path_mnist
    )

    sanity_check_no_nan(mnist_train, "MNIST train")
    sanity_check_no_nan(mnist_test, "MNIST test combined")
    sanity_check_same_width(mnist_train, mnist_test, "MNIST")

    save_no_header(mnist_train, os.path.join(SAVE_DIR, "mnist_train_processed.csv"))
    save_no_header(mnist_test, os.path.join(SAVE_DIR, "mnist_test_combined_processed.csv"))

    show_preview("MNIST train", mnist_train)
    show_preview("MNIST test combined", mnist_test)

    # -------------------------
    # SVHN
    # -------------------------
    svhn_train = read_headerless_logits(train_logit_path_svhn)
    svhn_test = combine_headerless_test_files(
        test_id_logit_path_svhn,
        test_ood_logit_path_svhn
    )

    sanity_check_no_nan(svhn_train, "SVHN train")
    sanity_check_no_nan(svhn_test, "SVHN test combined")
    sanity_check_same_width(svhn_train, svhn_test, "SVHN")

    save_no_header(svhn_train, os.path.join(SAVE_DIR, "svhn_train_processed.csv"))
    save_no_header(svhn_test, os.path.join(SAVE_DIR, "svhn_test_combined_processed.csv"))

    show_preview("SVHN train", svhn_train)
    show_preview("SVHN test combined", svhn_test)

    # -------------------------
    # MP
    # -------------------------
    mp_train, mp_train_meta = preprocess_mp(train_logit_path_mp10)
    mp_test, mp_test_meta = preprocess_mp(test_logit_path_mp10)

    sanity_check_no_nan(mp_train, "MP train")
    sanity_check_no_nan(mp_test, "MP test")
    sanity_check_same_width(mp_train, mp_test, "MP")

    save_no_header(mp_train, os.path.join(SAVE_DIR, "mp_train_processed.csv"))
    save_no_header(mp_test, os.path.join(SAVE_DIR, "mp_test_processed.csv"))

    save_json(mp_train_meta, os.path.join(SAVE_DIR, "mp_train_metadata.json"))
    save_json(mp_test_meta, os.path.join(SAVE_DIR, "mp_test_metadata.json"))

    show_preview("MP train", mp_train)
    show_preview("MP test", mp_test)

    print("\nMP train logit mapping:")
    for k, v in mp_train_meta["logit_index_to_canonical_label_name"].items():
        label_id = mp_train_meta["logit_index_to_canonical_label_id"][k]
        print(f"{k}: {v} (label_id={label_id})")

    print("\nMP test logit mapping:")
    for k, v in mp_test_meta["logit_index_to_canonical_label_name"].items():
        label_id = mp_test_meta["logit_index_to_canonical_label_id"][k]
        print(f"{k}: {v} (label_id={label_id})")

    # -------------------------
    # Tiny
    # -------------------------
    tiny_train = preprocess_tiny(train_logit_path_tiny)
    tiny_test = preprocess_tiny(test_logit_path_tiny)

    sanity_check_no_nan(tiny_train, "Tiny train")
    sanity_check_no_nan(tiny_test, "Tiny test")
    sanity_check_same_width(tiny_train, tiny_test, "Tiny")

    save_no_header(tiny_train, os.path.join(SAVE_DIR, "tiny_train_processed.csv"))
    save_no_header(tiny_test, os.path.join(SAVE_DIR, "tiny_test_processed.csv"))

    show_preview("Tiny train", tiny_train)
    show_preview("Tiny test", tiny_test)

    # -------------------------
    # summary metadata
    # -------------------------
    summary = {
        "output_format": "headerless CSV, first column is label, remaining columns are logits in file order",
        "saved_files": [
            "cifar_train_processed.csv",
            "cifar_test_combined_processed.csv",
            "cifar_train_first6_processed.csv",
            "cifar_test_first6_combined_processed.csv",
            "mnist_train_processed.csv",
            "mnist_test_combined_processed.csv",
            "svhn_train_processed.csv",
            "svhn_test_combined_processed.csv",
            "mp_train_processed.csv",
            "mp_test_processed.csv",
            "mp_train_metadata.json",
            "mp_test_metadata.json",
            "tiny_train_processed.csv",
            "tiny_test_processed.csv",
        ],
        "mp_canonical_label_order": MP_LABEL_ORDER,
    }
    save_json(summary, os.path.join(SAVE_DIR, "preprocessing_summary.json"))

    print("\nDone. Processed files saved to:", SAVE_DIR)


if __name__ == "__main__":
    main()


CIFAR train
shape: (18113, 5)
     0         1         2         3         4
0  3.0 -1.828775 -1.719470 -1.377560  4.878524
1  3.0 -1.479318 -1.774844 -1.434390  4.774768
2  2.0 -2.003274 -1.914887  4.682842 -1.530620
3  1.0 -1.559808  5.522950 -1.461287 -1.372268
4  3.0 -1.484001 -1.689840 -1.644417  4.717036

CIFAR test combined
shape: (14000, 5)
     0         1         2         3         4
0  3.0 -1.566677 -1.924289 -1.359065  4.819623
1  0.0  4.579016 -2.485156 -1.352865 -1.179378
2  1.0 -1.629191  5.556546 -1.585985 -1.348391
3  3.0 -1.899427 -1.728079 -1.271314  4.862905
4  1.0 -1.679571  5.494577 -1.356229 -1.318767

CIFAR first6 train
shape: (27083, 7)
     0         1         2         3         4         5         6
0  4.0 -1.017217 -1.543922 -0.326658 -1.522117  5.407427 -0.980998
1  3.0 -1.174644 -1.223317 -1.139580  5.062421 -1.401898 -0.665691
2  4.0 -1.781659 -2.283177 -1.800258 -0.343858  4.439189  1.102474
3  3.0 -0.919528 -1.413047 -1.162557  4.941039 -1.920506 -0.

In [9]:
import os
import json
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from scipy.stats import weibull_min
from scipy.special import softmax


# =========================================================
# Configuration
# =========================================================
BASE = "processed_logits"
THRESHOLD = 0.5

DATASETS = [
    {
        "name": "CIFAR",
        "train_path": os.path.join(BASE, "cifar_train_processed.csv"),
        "test_path": os.path.join(BASE, "cifar_test_combined_processed.csv"),
        "mp_metadata_path": None,
    },
    {
        "name": "CIFAR_first6",
        "train_path": os.path.join(BASE, "cifar_train_first6_processed.csv"),
        "test_path": os.path.join(BASE, "cifar_test_first6_combined_processed.csv"),
        "mp_metadata_path": None,
    },
    {
        "name": "MNIST",
        "train_path": os.path.join(BASE, "mnist_train_processed.csv"),
        "test_path": os.path.join(BASE, "mnist_test_combined_processed.csv"),
        "mp_metadata_path": None,
    },
    {
        "name": "SVHN",
        "train_path": os.path.join(BASE, "svhn_train_processed.csv"),
        "test_path": os.path.join(BASE, "svhn_test_combined_processed.csv"),
        "mp_metadata_path": None,
    },
    {
        "name": "MP",
        "train_path": os.path.join(BASE, "mp_train_processed.csv"),
        "test_path": os.path.join(BASE, "mp_test_processed.csv"),
        "mp_metadata_path": os.path.join(BASE, "mp_train_metadata.json"),
    },
    {
        "name": "Tiny",
        "train_path": os.path.join(BASE, "tiny_train_processed.csv"),
        "test_path": os.path.join(BASE, "tiny_test_processed.csv"),
        "mp_metadata_path": None,
    },
]


# =========================================================
# File loading
# =========================================================
def load_processed(path):
    """
    Processed file format:
        col 0: label
        col 1...: logits
    No header.
    """
    df = pd.read_csv(path, header=None)
    y = df.iloc[:, 0].to_numpy(dtype=int)
    X = df.iloc[:, 1:].to_numpy(dtype=float)
    return X, y


def load_mp_metadata(path):
    """
    MP metadata maps logit_i back to canonical label ids.
    """
    with open(path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    logit_map = meta["logit_index_to_canonical_label_id"]
    max_idx = max(int(k.split("_")[1]) for k in logit_map.keys())

    logit_index_to_label = np.zeros(max_idx + 1, dtype=int)
    for k, v in logit_map.items():
        i = int(k.split("_")[1])
        logit_index_to_label[i] = int(v)

    return meta, logit_index_to_label


# =========================================================
# OpenMax
# =========================================================
def eucos_distance(x, y, eu_scale=1.0, eu_weight=1.0, cos_weight=1.0, eps=1e-12):
    eu = np.linalg.norm(x - y) / max(eu_scale, eps)

    nx = np.linalg.norm(x)
    ny = np.linalg.norm(y)
    if nx < eps or ny < eps:
        cos_dist = 1.0
    else:
        cos_dist = cosine(x, y)

    return eu_weight * eu + cos_weight * cos_dist


class OpenMaxLogits:
    def __init__(self, tail_size=20, alpha=10, eu_weight=1.0, cos_weight=1.0):
        self.tail_size = tail_size
        self.alpha = alpha
        self.eu_weight = eu_weight
        self.cos_weight = cos_weight

        self.classes_ = None
        self.mavs = {}
        self.weibulls = {}
        self.eu_scales = {}

    def fit(self, train_logits, train_true, train_pred):
        """
        train_logits: shape (n_samples, n_logits)
        train_true: true labels in external label space
        train_pred: predicted labels in external label space
        """
        train_logits = np.asarray(train_logits, dtype=np.float64)
        train_true = np.asarray(train_true)
        train_pred = np.asarray(train_pred)

        self.classes_ = np.unique(train_true)

        min_needed = max(5, self.tail_size)

        for c in self.classes_:
            mask = (train_true == c) & (train_pred == c)
            class_logits = train_logits[mask]

            if len(class_logits) < min_needed:
                raise ValueError(
                    f"Not enough correctly classified training examples for class {c}. "
                    f"Need at least {min_needed}, got {len(class_logits)}."
                )

            mav = class_logits.mean(axis=0)
            self.mavs[c] = mav

            eu_dists = np.array(
                [np.linalg.norm(z - mav) for z in class_logits],
                dtype=np.float64
            )
            eu_scale = np.median(eu_dists) + 1e-12
            self.eu_scales[c] = eu_scale

            dists = np.array([
                eucos_distance(
                    z, mav,
                    eu_scale=eu_scale,
                    eu_weight=self.eu_weight,
                    cos_weight=self.cos_weight
                )
                for z in class_logits
            ], dtype=np.float64)

            tail = np.sort(dists)[-min(self.tail_size, len(dists)):]
            shape, loc, scale = weibull_min.fit(tail, floc=0.0)
            self.weibulls[c] = (shape, loc, scale)

        return self

    def _weibull_cdf(self, c, dist):
        shape, loc, scale = self.weibulls[c]
        return weibull_min.cdf(dist, c=shape, loc=loc, scale=scale)

    def recalibrate(self, x, logit_index_to_label=None):
        """
        x: shape (n_logits,)

        Returns:
            probs: shape (n_logits + 1,)
            index 0 = unknown
            indices 1... = logit_0, logit_1, ...
        """
        x = np.asarray(x, dtype=np.float64)
        n_logits = x.shape[0]

        ranked = np.argsort(x)[::-1]
        topk = ranked[:min(self.alpha, n_logits)]

        omega = np.ones(n_logits, dtype=np.float64)
        alpha_k = min(self.alpha, n_logits)

        for rank_pos, logit_idx in enumerate(topk):
            class_label = logit_idx if logit_index_to_label is None else int(logit_index_to_label[logit_idx])

            if class_label not in self.mavs:
                continue

            alpha_weight = (alpha_k - rank_pos) / alpha_k

            dist = eucos_distance(
                x,
                self.mavs[class_label],
                eu_scale=self.eu_scales[class_label],
                eu_weight=self.eu_weight,
                cos_weight=self.cos_weight
            )
            wscore = self._weibull_cdf(class_label, dist)
            omega[logit_idx] = 1.0 - alpha_weight * wscore

        revised_logits = x * omega
        unknown_logit = np.sum(x * (1.0 - omega))
        logits_with_unknown = np.concatenate([[unknown_logit], revised_logits])
        probs = softmax(logits_with_unknown)

        return probs

    def predict(self, test_logits, threshold=0.5, logit_index_to_label=None):
        """
        Returns:
            preds: external label ids, or -1 for unknown
            probs_all: OpenMax probabilities
        """
        test_logits = np.asarray(test_logits, dtype=np.float64)

        preds = []
        probs_all = []

        for x in test_logits:
            probs = self.recalibrate(x, logit_index_to_label=logit_index_to_label)
            probs_all.append(probs)

            best_idx = int(np.argmax(probs))
            best_prob = float(probs[best_idx])

            if best_idx == 0:
                preds.append(-1)
            else:
                logit_idx = best_idx - 1
                pred_label = logit_idx if logit_index_to_label is None else int(logit_index_to_label[logit_idx])

                if threshold is not None and best_prob < threshold:
                    preds.append(-1)
                else:
                    preds.append(pred_label)

        return np.array(preds, dtype=int), np.array(probs_all)


# =========================================================
# Evaluation
# =========================================================
def compute_fscore_from_train_labels(y_true, y_pred, train_labels):
    """
    ID classes are the labels that appear in the training set.
    Any test label not in the training set is treated as OOD.

    This implements the user's desired logic, but without assuming
    ID labels are ordered as 0..K-1.

    Evaluation space:
        [ID class 0, ID class 1, ..., ID class K-1, OOD]

    Returns:
        dict with macro_f1, per-class F1, TP/FP/FN, and mappings
    """
    train_labels = sorted(set(int(x) for x in train_labels))
    id_class_to_index = {label: i for i, label in enumerate(train_labels)}
    index_to_id_class = {i: label for label, i in id_class_to_index.items()}

    ood_index = len(train_labels)
    num_eval_classes = len(train_labels) + 1

    # Map true labels into eval index space
    eval_true = np.array([
        id_class_to_index[y] if y in id_class_to_index else ood_index
        for y in y_true
    ], dtype=int)

    # Map predictions into eval index space
    eval_pred = []
    for p in y_pred:
        if p == -1:
            eval_pred.append(ood_index)
        elif p in id_class_to_index:
            eval_pred.append(id_class_to_index[p])
        else:
            # Any predicted label outside the training ID label set is treated as OOD bucket
            eval_pred.append(ood_index)
    eval_pred = np.array(eval_pred, dtype=int)

    truePos = np.zeros(num_eval_classes, dtype=float)
    falsePos = np.zeros(num_eval_classes, dtype=float)
    falseNeg = np.zeros(num_eval_classes, dtype=float)

    for trueClass, predClass in zip(eval_true, eval_pred):
        # OOD true class
        if trueClass == ood_index:
            if predClass == ood_index:
                truePos[trueClass] += 1
            else:
                falseNeg[trueClass] += 1
                falsePos[predClass] += 1
        # ID true class
        else:
            if predClass == trueClass:
                truePos[trueClass] += 1
            else:
                falseNeg[trueClass] += 1
                falsePos[predClass] += 1

    precision = truePos / (truePos + falsePos + 1e-12)
    recall = truePos / (truePos + falseNeg + 1e-12)
    f1_per_class = 2 * precision * recall / (precision + recall + 1e-12)
    macro_f1 = float(np.mean(f1_per_class))

    class_names = []
    for i in range(num_eval_classes):
        if i == ood_index:
            class_names.append("OOD")
        else:
            class_names.append(f"ID_label_{index_to_id_class[i]}")

    return {
        "macro_f1": macro_f1,
        "f1_per_class": f1_per_class,
        "precision_per_class": precision,
        "recall_per_class": recall,
        "truePos": truePos,
        "falsePos": falsePos,
        "falseNeg": falseNeg,
        "id_class_to_eval_index": id_class_to_index,
        "eval_index_to_id_class": index_to_id_class,
        "ood_eval_index": ood_index,
        "class_names": class_names,
        "eval_true": eval_true,
        "eval_pred": eval_pred,
    }


def print_eval_results(dataset_name, results):
    print(f"\n{'=' * 90}")
    print(f"{dataset_name} RESULTS")
    print(f"{'=' * 90}")
    print(f"Macro F1: {results['macro_f1']:.6f}\n")

    print("Per-class results:")
    for i, class_name in enumerate(results["class_names"]):
        print(
            f"{class_name:>15} | "
            f"F1={results['f1_per_class'][i]:.6f} | "
            f"P={results['precision_per_class'][i]:.6f} | "
            f"R={results['recall_per_class'][i]:.6f} | "
            f"TP={int(results['truePos'][i])} | "
            f"FP={int(results['falsePos'][i])} | "
            f"FN={int(results['falseNeg'][i])}"
        )


# =========================================================
# End-to-end runner
# =========================================================
def run_dataset(name, train_path, test_path, threshold=0.5, mp_metadata_path=None):
    print(f"\nRunning {name}...")
    print(f"Train file: {train_path}")
    print(f"Test file : {test_path}")

    # 1. Load files
    X_train, y_train = load_processed(train_path)
    X_test, y_test = load_processed(test_path)

    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_test shape :", X_test.shape)
    print("y_test shape :", y_test.shape)

    # 2. Load MP mapping if needed
    logit_index_to_label = None
    if mp_metadata_path is not None:
        _, logit_index_to_label = load_mp_metadata(mp_metadata_path)
        print("Loaded MP metadata mapping from:", mp_metadata_path)

    # 3. Base model predictions on training set
    #    These must be in external label space
    if logit_index_to_label is None:
        train_pred = np.argmax(X_train, axis=1).astype(int)
    else:
        train_pred = np.array(
            [int(logit_index_to_label[i]) for i in np.argmax(X_train, axis=1)],
            dtype=int
        )

    # 4. Fit OpenMax
    om = OpenMaxLogits(
        tail_size=20,
        alpha=min(10, X_train.shape[1]),
        eu_weight=1.0,
        cos_weight=1.0,
    )
    om.fit(X_train, y_train, train_pred)

    # 5. Predict on test set
    preds, probs = om.predict(
        X_test,
        threshold=threshold,
        logit_index_to_label=logit_index_to_label
    )

    # 6. Evaluate with ID classes = labels in training set
    eval_results = compute_fscore_from_train_labels(
        y_true=y_test,
        y_pred=preds,
        train_labels=y_train,
    )

    print_eval_results(name, eval_results)

    return {
        "dataset": name,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "train_pred": train_pred,
        "preds": preds,
        "probs": probs,
        "eval": eval_results,
    }


def main():
    all_results = {}

    for cfg in DATASETS:
        result = run_dataset(
            name=cfg["name"],
            train_path=cfg["train_path"],
            test_path=cfg["test_path"],
            threshold=THRESHOLD,
            mp_metadata_path=cfg["mp_metadata_path"],
        )
        all_results[cfg["name"]] = result

    print(f"\n{'#' * 90}")
    print("SUMMARY")
    print(f"{'#' * 90}")
    for name, result in all_results.items():
        print(f"{name:>8} | Macro F1 = {result['eval']['macro_f1']:.6f}")

    return all_results


if __name__ == "__main__":
    all_results = main()


Running CIFAR...
Train file: processed_logits/cifar_train_processed.csv
Test file : processed_logits/cifar_test_combined_processed.csv
X_train shape: (18113, 4)
y_train shape: (18113,)
X_test shape : (14000, 4)
y_test shape : (14000,)

CIFAR RESULTS
Macro F1: 0.590266

Per-class results:
     ID_label_0 | F1=0.675510 | P=0.511856 | R=0.993000 | TP=993 | FP=947 | FN=7
     ID_label_1 | F1=0.846939 | P=0.736686 | R=0.996000 | TP=996 | FP=356 | FN=4
     ID_label_2 | F1=0.460745 | P=0.300888 | R=0.983000 | TP=983 | FP=2284 | FN=17
     ID_label_3 | F1=0.423449 | P=0.269620 | R=0.986000 | TP=986 | FP=2671 | FN=14
            OOD | F1=0.544689 | P=0.992072 | R=0.375400 | TP=3754 | FP=30 | FN=6246

Running CIFAR_first6...
Train file: processed_logits/cifar_train_first6_processed.csv
Test file : processed_logits/cifar_test_first6_combined_processed.csv
X_train shape: (18113, 4)
y_train shape: (18113,)
X_test shape : (14000, 4)
y_test shape : (14000,)

CIFAR_first6 RESULTS
Macro F1: 0.590266


In [10]:
import os
import json
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from scipy.stats import weibull_min
from scipy.special import softmax


# =========================================================
# Configuration
# =========================================================
BASE = "processed_logits"
THRESHOLD = 0

DATASETS = [
    {
        "name": "CIFAR",
        "train_path": os.path.join(BASE, "cifar_train_processed.csv"),
        "test_path": os.path.join(BASE, "cifar_test_combined_processed.csv"),
        "mp_metadata_path": None,
    },
    {
        "name": "CIFAR_first6",
        "train_path": os.path.join(BASE, "cifar_train_first6_processed.csv"),
        "test_path": os.path.join(BASE, "cifar_test_first6_combined_processed.csv"),
        "mp_metadata_path": None,
    },
    {
        "name": "MNIST",
        "train_path": os.path.join(BASE, "mnist_train_processed.csv"),
        "test_path": os.path.join(BASE, "mnist_test_combined_processed.csv"),
        "mp_metadata_path": None,
    },
    {
        "name": "SVHN",
        "train_path": os.path.join(BASE, "svhn_train_processed.csv"),
        "test_path": os.path.join(BASE, "svhn_test_combined_processed.csv"),
        "mp_metadata_path": None,
    },
    {
        "name": "MP",
        "train_path": os.path.join(BASE, "mp_train_processed.csv"),
        "test_path": os.path.join(BASE, "mp_test_processed.csv"),
        "mp_metadata_path": os.path.join(BASE, "mp_train_metadata.json"),
    },
    {
        "name": "Tiny",
        "train_path": os.path.join(BASE, "tiny_train_processed.csv"),
        "test_path": os.path.join(BASE, "tiny_test_processed.csv"),
        "mp_metadata_path": None,
    },
]


# =========================================================
# File loading
# =========================================================
def load_processed(path):
    """
    Processed file format:
        col 0: label
        col 1...: logits
    No header.
    """
    df = pd.read_csv(path, header=None)
    y = df.iloc[:, 0].to_numpy(dtype=int)
    X = df.iloc[:, 1:].to_numpy(dtype=float)
    return X, y


def load_mp_metadata(path):
    """
    MP metadata maps logit_i back to canonical label ids.
    """
    with open(path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    logit_map = meta["logit_index_to_canonical_label_id"]
    max_idx = max(int(k.split("_")[1]) for k in logit_map.keys())

    logit_index_to_label = np.zeros(max_idx + 1, dtype=int)
    for k, v in logit_map.items():
        i = int(k.split("_")[1])
        logit_index_to_label[i] = int(v)

    return meta, logit_index_to_label


# =========================================================
# OpenMax
# =========================================================
def eucos_distance(x, y, eu_scale=1.0, eu_weight=1.0, cos_weight=1.0, eps=1e-12):
    eu = np.linalg.norm(x - y) / max(eu_scale, eps)

    nx = np.linalg.norm(x)
    ny = np.linalg.norm(y)
    if nx < eps or ny < eps:
        cos_dist = 1.0
    else:
        cos_dist = cosine(x, y)

    return eu_weight * eu + cos_weight * cos_dist


class OpenMaxLogits:
    def __init__(self, tail_size=20, alpha=10, eu_weight=1.0, cos_weight=1.0):
        self.tail_size = tail_size
        self.alpha = alpha
        self.eu_weight = eu_weight
        self.cos_weight = cos_weight

        self.classes_ = None
        self.mavs = {}
        self.weibulls = {}
        self.eu_scales = {}

    def fit(self, train_logits, train_true, train_pred):
        """
        train_logits: shape (n_samples, n_logits)
        train_true: true labels in external label space
        train_pred: predicted labels in external label space
        """
        train_logits = np.asarray(train_logits, dtype=np.float64)
        train_true = np.asarray(train_true)
        train_pred = np.asarray(train_pred)

        self.classes_ = np.unique(train_true)

        min_needed = max(5, self.tail_size)

        for c in self.classes_:
            mask = (train_true == c) & (train_pred == c)
            class_logits = train_logits[mask]

            if len(class_logits) < min_needed:
                raise ValueError(
                    f"Not enough correctly classified training examples for class {c}. "
                    f"Need at least {min_needed}, got {len(class_logits)}."
                )

            mav = class_logits.mean(axis=0)
            self.mavs[c] = mav

            eu_dists = np.array(
                [np.linalg.norm(z - mav) for z in class_logits],
                dtype=np.float64
            )
            eu_scale = np.median(eu_dists) + 1e-12
            self.eu_scales[c] = eu_scale

            dists = np.array([
                eucos_distance(
                    z, mav,
                    eu_scale=eu_scale,
                    eu_weight=self.eu_weight,
                    cos_weight=self.cos_weight
                )
                for z in class_logits
            ], dtype=np.float64)

            tail = np.sort(dists)[-min(self.tail_size, len(dists)):]
            shape, loc, scale = weibull_min.fit(tail, floc=0.0)
            self.weibulls[c] = (shape, loc, scale)

        return self

    def _weibull_cdf(self, c, dist):
        shape, loc, scale = self.weibulls[c]
        return weibull_min.cdf(dist, c=shape, loc=loc, scale=scale)

    def recalibrate(self, x, logit_index_to_label=None):
        """
        x: shape (n_logits,)

        Returns:
            probs: shape (n_logits + 1,)
            index 0 = unknown
            indices 1... = logit_0, logit_1, ...
        """
        x = np.asarray(x, dtype=np.float64)
        n_logits = x.shape[0]

        ranked = np.argsort(x)[::-1]
        topk = ranked[:min(self.alpha, n_logits)]

        omega = np.ones(n_logits, dtype=np.float64)
        alpha_k = min(self.alpha, n_logits)

        for rank_pos, logit_idx in enumerate(topk):
            class_label = logit_idx if logit_index_to_label is None else int(logit_index_to_label[logit_idx])

            if class_label not in self.mavs:
                continue

            alpha_weight = (alpha_k - rank_pos) / alpha_k

            dist = eucos_distance(
                x,
                self.mavs[class_label],
                eu_scale=self.eu_scales[class_label],
                eu_weight=self.eu_weight,
                cos_weight=self.cos_weight
            )
            wscore = self._weibull_cdf(class_label, dist)
            omega[logit_idx] = 1.0 - alpha_weight * wscore

        revised_logits = x * omega
        unknown_logit = np.sum(x * (1.0 - omega))
        logits_with_unknown = np.concatenate([[unknown_logit], revised_logits])
        probs = softmax(logits_with_unknown)

        return probs

    def predict(self, test_logits, threshold=0.5, logit_index_to_label=None):
        """
        Returns:
            preds: external label ids, or -1 for unknown
            probs_all: OpenMax probabilities
        """
        test_logits = np.asarray(test_logits, dtype=np.float64)

        preds = []
        probs_all = []

        for x in test_logits:
            probs = self.recalibrate(x, logit_index_to_label=logit_index_to_label)
            probs_all.append(probs)

            best_idx = int(np.argmax(probs))
            best_prob = float(probs[best_idx])

            if best_idx == 0:
                preds.append(-1)
            else:
                logit_idx = best_idx - 1
                pred_label = logit_idx if logit_index_to_label is None else int(logit_index_to_label[logit_idx])

                if threshold is not None and best_prob < threshold:
                    preds.append(-1)
                else:
                    preds.append(pred_label)

        return np.array(preds, dtype=int), np.array(probs_all)


# =========================================================
# Evaluation
# =========================================================
def compute_fscore_from_train_labels(y_true, y_pred, train_labels):
    """
    ID classes are labels that appear in the training set.
    Unknown/OOD classes are labels in y_true that do NOT appear in training.

    Unlike the previous version, OOD classes are NOT aggregated.
    Each unknown true label gets its own evaluation class.

    Decision logic:
      - If true label is ID:
            correct iff pred == true label
      - If true label is OOD:
            correct iff pred == -1

        When pred == -1 for an OOD sample, we count TP for that sample's
        specific unknown class.

      - If pred == -1 for an ID sample:
            count FN for the true ID class.
            There is no single aggregated OOD FP bucket here.

    Evaluation space:
        [ID labels..., OOD label A, OOD label B, ...]

    Returns:
        macro_f1, per-class F1, TP/FP/FN, mappings, eval_true, eval_pred
    """
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    id_labels = sorted(set(int(x) for x in train_labels))
    id_label_set = set(id_labels)

    ood_labels = sorted(set(int(y) for y in y_true if int(y) not in id_label_set))

    eval_labels = id_labels + ood_labels
    label_to_eval_index = {label: i for i, label in enumerate(eval_labels)}
    eval_index_to_label = {i: label for label, i in label_to_eval_index.items()}

    num_eval_classes = len(eval_labels)

    truePos = np.zeros(num_eval_classes, dtype=float)
    falsePos = np.zeros(num_eval_classes, dtype=float)
    falseNeg = np.zeros(num_eval_classes, dtype=float)

    eval_true = []
    eval_pred = []

    for t, p in zip(y_true, y_pred):
        t = int(t)
        p = int(p)

        true_idx = label_to_eval_index[t]
        eval_true.append(true_idx)

        # -------------------------------------------------
        # Case 1: true sample is OOD
        # -------------------------------------------------
        if t not in id_label_set:
            # OpenMax rejection means correctly detected as this OOD class
            if p == -1:
                pred_idx = true_idx
                truePos[true_idx] += 1
            else:
                falseNeg[true_idx] += 1

                # If accepted as an ID class, FP for that ID class
                if p in label_to_eval_index:
                    pred_idx = label_to_eval_index[p]
                    falsePos[pred_idx] += 1
                else:
                    # predicted some label outside eval space;
                    # no valid class bucket, so track as -1
                    pred_idx = -1

        # -------------------------------------------------
        # Case 2: true sample is ID
        # -------------------------------------------------
        else:
            if p == t:
                pred_idx = true_idx
                truePos[true_idx] += 1
            else:
                falseNeg[true_idx] += 1

                if p == -1:
                    # ID rejected as unknown:
                    # FN for true ID class.
                    # No aggregated unknown FP bucket because OOD is not aggregated.
                    pred_idx = -1
                elif p in label_to_eval_index:
                    pred_idx = label_to_eval_index[p]
                    falsePos[pred_idx] += 1
                else:
                    pred_idx = -1

        eval_pred.append(pred_idx)

    eval_true = np.array(eval_true, dtype=int)
    eval_pred = np.array(eval_pred, dtype=int)

    precision = truePos / (truePos + falsePos + 1e-12)
    recall = truePos / (truePos + falseNeg + 1e-12)
    f1_per_class = 2 * precision * recall / (precision + recall + 1e-12)
    macro_f1 = float(np.mean(f1_per_class))

    class_names = []
    for label in eval_labels:
        if label in id_label_set:
            class_names.append(f"ID_label_{label}")
        else:
            class_names.append(f"OOD_label_{label}")

    return {
        "macro_f1": macro_f1,
        "f1_per_class": f1_per_class,
        "precision_per_class": precision,
        "recall_per_class": recall,
        "truePos": truePos,
        "falsePos": falsePos,
        "falseNeg": falseNeg,
        "id_labels": id_labels,
        "ood_labels": ood_labels,
        "eval_labels": eval_labels,
        "label_to_eval_index": label_to_eval_index,
        "eval_index_to_label": eval_index_to_label,
        "class_names": class_names,
        "eval_true": eval_true,
        "eval_pred": eval_pred,
    }


def print_eval_results(dataset_name, results):
    print(f"\n{'=' * 90}")
    print(f"{dataset_name} RESULTS")
    print(f"{'=' * 90}")
    print(f"Macro F1: {results['macro_f1']:.6f}\n")

    print("Per-class results:")
    for i, class_name in enumerate(results["class_names"]):
        print(
            f"{class_name:>15} | "
            f"F1={results['f1_per_class'][i]:.6f} | "
            f"P={results['precision_per_class'][i]:.6f} | "
            f"R={results['recall_per_class'][i]:.6f} | "
            f"TP={int(results['truePos'][i])} | "
            f"FP={int(results['falsePos'][i])} | "
            f"FN={int(results['falseNeg'][i])}"
        )


# =========================================================
# End-to-end runner
# =========================================================
def run_dataset(name, train_path, test_path, threshold=0.5, mp_metadata_path=None):
    print(f"\nRunning {name}...")
    print(f"Train file: {train_path}")
    print(f"Test file : {test_path}")

    # 1. Load files
    X_train, y_train = load_processed(train_path)
    X_test, y_test = load_processed(test_path)

    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_test shape :", X_test.shape)
    print("y_test shape :", y_test.shape)

    # 2. Load MP mapping if needed
    logit_index_to_label = None
    if mp_metadata_path is not None:
        _, logit_index_to_label = load_mp_metadata(mp_metadata_path)
        print("Loaded MP metadata mapping from:", mp_metadata_path)

    # 3. Base model predictions on training set
    #    These must be in external label space
    if logit_index_to_label is None:
        train_pred = np.argmax(X_train, axis=1).astype(int)
    else:
        train_pred = np.array(
            [int(logit_index_to_label[i]) for i in np.argmax(X_train, axis=1)],
            dtype=int
        )

    # 4. Fit OpenMax
    om = OpenMaxLogits(
        tail_size=20,
        alpha=min(10, X_train.shape[1]),
        eu_weight=1.0,
        cos_weight=1.0,
    )
    om.fit(X_train, y_train, train_pred)

    # 5. Predict on test set
    preds, probs = om.predict(
        X_test,
        threshold=threshold,
        logit_index_to_label=logit_index_to_label
    )

    # 6. Evaluate with ID classes = labels in training set
    eval_results = compute_fscore_from_train_labels(
        y_true=y_test,
        y_pred=preds,
        train_labels=y_train,
    )

    print_eval_results(name, eval_results)

    return {
        "dataset": name,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "train_pred": train_pred,
        "preds": preds,
        "probs": probs,
        "eval": eval_results,
    }


def main():
    all_results = {}

    for cfg in DATASETS:
        result = run_dataset(
            name=cfg["name"],
            train_path=cfg["train_path"],
            test_path=cfg["test_path"],
            threshold=THRESHOLD,
            mp_metadata_path=cfg["mp_metadata_path"],
        )
        all_results[cfg["name"]] = result

    print(f"\n{'#' * 90}")
    print("SUMMARY")
    print(f"{'#' * 90}")
    for name, result in all_results.items():
        print(f"{name:>8} | Macro F1 = {result['eval']['macro_f1']:.6f}")

    return all_results


if __name__ == "__main__":
    all_results = main()


Running CIFAR...
Train file: processed_logits/cifar_train_processed.csv
Test file : processed_logits/cifar_test_combined_processed.csv
X_train shape: (18113, 4)
y_train shape: (18113,)
X_test shape : (14000, 4)
y_test shape : (14000,)

CIFAR RESULTS
Macro F1: 0.407739

Per-class results:
     ID_label_0 | F1=0.644200 | P=0.476510 | R=0.994000 | TP=994 | FP=1092 | FN=6
     ID_label_1 | F1=0.837327 | P=0.722263 | R=0.996000 | TP=996 | FP=383 | FN=4
     ID_label_2 | F1=0.419595 | P=0.266576 | R=0.985000 | TP=985 | FP=2710 | FN=15
     ID_label_3 | F1=0.392211 | P=0.244731 | R=0.987000 | TP=987 | FP=3046 | FN=13
    OOD_label_4 | F1=0.571429 | P=1.000000 | R=0.400000 | TP=40 | FP=0 | FN=60
    OOD_label_5 | F1=0.400000 | P=1.000000 | R=0.250000 | TP=25 | FP=0 | FN=75
    OOD_label_6 | F1=0.181818 | P=1.000000 | R=0.100000 | TP=10 | FP=0 | FN=90
    OOD_label_7 | F1=0.181818 | P=1.000000 | R=0.100000 | TP=10 | FP=0 | FN=90
    OOD_label_8 | F1=0.165138 | P=1.000000 | R=0.090000 | TP=9 | 

In [11]:
all_results['MNIST']['eval']['f1_per_class']

array([0.83461211, 0.87446891, 0.77946768, 0.88859416, 0.66802999,
       0.76326003, 0.51126651, 0.2923588 , 0.1777362 , 0.51179941])

In [12]:
all_results.keys()

dict_keys(['CIFAR', 'CIFAR_first6', 'MNIST', 'SVHN', 'MP', 'Tiny'])

In [29]:
def compute_means(score_array, id_classes):
    """
    Args:
        score_dict (dict): {class_index: value}
        id_classes (int): number of ID classes (assumed to be first indices)

    Returns:
        tuple:
            (mean_id, mean_ood, mean_bucketed)
    """
    # sort by key to ensure correct order
    values = score_array

    # split
    id_vals = values[:id_classes]
    ood_vals = values[id_classes:]

    # compute means
    mean_id = sum(id_vals) / len(id_vals) if len(id_vals) > 0 else 0.0
    mean_ood = sum(ood_vals) / len(ood_vals) if len(ood_vals) > 0 else 0.0

    # bucketed: ID classes + 1 class (mean of OOD)
    #print(id_vals)
    
    bucketed_vals = np.concatenate([id_vals, [mean_ood]])
    #print(len(bucketed_vals) )
    #print(bucketed_vals)
    mean_bucketed = sum(bucketed_vals) / len(bucketed_vals) if len(bucketed_vals) > 0 else 0.0

    return mean_id, mean_ood, mean_bucketed

In [41]:
print(f"MNIST OpenMax: {compute_means(all_results['MNIST']['eval']['f1_per_class'],6)}")
print(f"CIFAR10 OpenMax: {compute_means(all_results['CIFAR_first6']['eval']['f1_per_class'],6)}")
print(f"TINY OpenMax: {compute_means(all_results['Tiny']['eval']['f1_per_class'],20)}")
print(f"MP10 OpenMax: {compute_means(all_results['MP']['eval']['f1_per_class'],10)}")
print(f"SVHN OpenMax: {compute_means(all_results['SVHN']['eval']['f1_per_class'],6)}")

MNIST OpenMax: (0.8014054794573385, 0.37329023183488586, 0.7402461583684167)
CIFAR10 OpenMax: (0.544126918214095, 0.39938921309052217, 0.5234501031964417)
TINY OpenMax: (0.2403575812815391, 0.7414344309290688, 0.26421838364570716)
MP10 OpenMax: (0.6820533093298131, 0.9124747288468629, 0.7030007111040905)
SVHN OpenMax: (0.8030323171473838, 0.0002527805864507077, 0.6883495262101075)


In [36]:
all_results['CIFAR']['eval']['f1_per_class']

104

In [39]:
import numpy as np

# y_values: length 104 (first 4 = known, rest = unknown)
y_values = np.array(all_results['CIFAR']['eval']['f1_per_class'])

known_f = y_values[:4]
unknown_f = y_values[4:]

def sample_stats_with_bucket(known_f, unknown_f, subset_size, n_samples=200, seed=42):
    rng = np.random.default_rng(seed)

    known_mean = np.mean(known_f)

    unknown_sample_means = []
    bucketed_means = []

    for _ in range(n_samples):
        sampled_unknowns = rng.choice(unknown_f, size=subset_size, replace=False)

        unknown_mean = np.mean(sampled_unknowns)

        # 4 known classes + 1 unknown bucket
        bucketed = np.concatenate([known_f, [unknown_mean]])
        bucketed_mean = np.mean(bucketed)

        unknown_sample_means.append(unknown_mean)
        bucketed_means.append(bucketed_mean)

    unknown_sample_means = np.array(unknown_sample_means)
    bucketed_means = np.array(bucketed_means)

    return {
        "known_mean": known_mean,

        "unknown_mean": unknown_sample_means.mean(),
        "unknown_std": unknown_sample_means.std(ddof=1),

        "bucketed_mean": bucketed_means.mean(),
        "bucketed_var": bucketed_means.std(ddof=1),

        "unknown_all_samples": unknown_sample_means,
        "bucketed_all_samples": bucketed_means,
    }

stats_10 = sample_stats_with_bucket(
    known_f, unknown_f, subset_size=10, n_samples=200, seed=42
)

stats_50 = sample_stats_with_bucket(
    known_f, unknown_f, subset_size=50, n_samples=200, seed=42
)

print("CIFAR+10")
print("Known mean:", stats_10["known_mean"])
print("Unknown mean:", stats_10["unknown_mean"])
print("Unknown std:", stats_10["unknown_std"])
print("Bucketed mean:", stats_10["bucketed_mean"])
print("Bucketed std:", stats_10["bucketed_var"])

print("\nCIFAR+50")
print("Known mean:", stats_50["known_mean"])
print("Unknown mean:", stats_50["unknown_mean"])
print("Unknown std:", stats_50["unknown_std"])
print("Bucketed mean:", stats_50["bucketed_mean"])
print("Bucketed std:", stats_50["bucketed_var"])

CIFAR+10
Known mean: 0.5733332344641853
Unknown mean: 0.3973832668547513
Unknown std: 0.06776322964418456
Bucketed mean: 0.5381432409422985
Bucketed std: 0.013552645928836912

CIFAR+50
Known mean: 0.5733332344641853
Unknown mean: 0.3987189071840049
Unknown std: 0.022786812708359746
Bucketed mean: 0.5384103690081493
Bucketed std: 0.00455736254167195
